# Open camera Image Metadata Extractor

This notebook reads one image captured by the Mapillary app and extracts:
- yaw
- pitch
- roll
- gps (lat, lon, alt)
- image width, image height
- focal_length_mm
- sensor_width_mm
- sensor_height_mm
- focal_length_35mm_equiv

It checks both EXIF and embedded XMP metadata because different devices/apps store fields in different places.

In [1]:
import json
import re
from pathlib import Path
from typing import Any
from xml.etree import ElementTree as ET

from PIL import ExifTags, Image

EXIF_TAGS = ExifTags.TAGS
GPS_TAGS = ExifTags.GPSTAGS

In [2]:
def rational_to_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        pass

    if isinstance(value, tuple) and len(value) == 2:
        num, den = value
        den = float(den)
        if den == 0:
            return None
        return float(num) / den

    if hasattr(value, 'numerator') and hasattr(value, 'denominator'):
        den = float(value.denominator)
        if den == 0:
            return None
        return float(value.numerator) / den

    return None


def dms_to_decimal(dms: Any, ref: str | None) -> float | None:
    if dms is None:
        return None

    try:
        d = rational_to_float(dms[0])
        m = rational_to_float(dms[1])
        s = rational_to_float(dms[2])
    except Exception:
        return None

    if d is None or m is None or s is None:
        return None

    val = d + (m / 60.0) + (s / 3600.0)
    if ref in {'S', 'W'}:
        val = -val
    return val


def decode_if_bytes(value: Any) -> Any:
    if isinstance(value, bytes):
        for enc in ('utf-8', 'latin-1'):
            try:
                return value.decode(enc, errors='ignore')
            except Exception:
                continue
    return value


def get_exif_dict(image_path: Path) -> dict[str, Any]:
    """
    FIX 1: Use get_ifd(ExifTags.IFD.GPSInfo) to decode GPS sub-IFD.
    FIX 2: Use get_ifd(ExifTags.IFD.Exif) to decode Exif sub-IFD.
    In Pillow, raw_exif.items() only returns top-level IFD0 tags.
    Critical tags like FocalLength, FocalLengthIn35mmFilm, and UserComment
    live in the Exif sub-IFD and are silently missed without this fix.
    """
    image = Image.open(image_path)
    raw_exif = image.getexif()
    exif_named = {}

    # Top-level IFD0 tags (skip sub-IFD placeholders which are raw int offsets)
    skip_tag_names = {'GPSInfo', 'ExifOffset'}
    for tag_id, value in raw_exif.items():
        tag_name = EXIF_TAGS.get(tag_id, str(tag_id))
        if tag_name in skip_tag_names:
            continue
        exif_named[tag_name] = decode_if_bytes(value)

    # FIX: Decode Exif sub-IFD (FocalLength, UserComment, ISO, etc.)
    try:
        exif_ifd = raw_exif.get_ifd(ExifTags.IFD.Exif)
        for tag_id, value in exif_ifd.items():
            tag_name = EXIF_TAGS.get(tag_id, str(tag_id))
            exif_named.setdefault(tag_name, decode_if_bytes(value))
    except Exception:
        pass

    # FIX: Decode GPS sub-IFD properly
    try:
        gps_ifd = raw_exif.get_ifd(ExifTags.IFD.GPSInfo)
        if gps_ifd:
            gps_named = {}
            for gps_id, gps_val in gps_ifd.items():
                gps_name = GPS_TAGS.get(gps_id, str(gps_id))
                gps_named[gps_name] = decode_if_bytes(gps_val)
            exif_named['GPSInfo'] = gps_named
    except Exception:
        pass

    return exif_named


def extract_xmp_text(image_path: Path) -> str:
    data = image_path.read_bytes()
    start = data.find(b'<x:xmpmeta')
    if start == -1:
        start = data.find(b'<rdf:RDF')
        if start == -1:
            return ''

    end = data.find(b'</x:xmpmeta>')
    if end != -1:
        end += len(b'</x:xmpmeta>')
    else:
        end = data.find(b'</rdf:RDF>')
        if end != -1:
            end += len(b'</rdf:RDF>')
        else:
            end = min(len(data), start + 200000)

    blob = data[start:end]
    return blob.decode('utf-8', errors='ignore')


def parse_xmp_attributes(xmp_text: str) -> dict[str, str]:
    if not xmp_text.strip():
        return {}

    attrs = {}

    # Fast regex fallback for all key="value" pairs.
    for key, val in re.findall(r'([A-Za-z0-9:_.-]+)\s*=\s*"([^"]+)"', xmp_text):
        attrs.setdefault(key, val)

    # Try XML parser too; merge discovered attributes.
    try:
        root = ET.fromstring(xmp_text)
        for elem in root.iter():
            for k, v in elem.attrib.items():
                short_k = k.split('}', 1)[-1]
                attrs.setdefault(short_k, v)
    except Exception:
        pass

    return attrs


In [3]:
def pick_float(candidates: list[str], source: dict[str, Any]) -> float | None:
    for key in candidates:
        if key in source and source[key] not in (None, ''):
            raw = source[key]
            try:
                out = float(raw)
                if out != out:
                    continue
                return out
            except Exception:
                m = re.search(r'-?\d+(?:\.\d+)?', str(raw))
                if m:
                    try:
                        return float(m.group(0))
                    except Exception:
                        pass
                continue
    return None


def parse_user_comment_for_pose(comment: Any) -> tuple[float | None, float | None, float | None]:
    """
    Strictly parse UserComment that looks like:
      b'ASCII\x00\x00\x00Yaw:319.4704,Pitch:1.5786,Roll:-0.4576'
    Returns (yaw, pitch, roll) as floats or (None, None, None).
    """
    if comment is None:
        return (None, None, None)
    # normalize to string and strip common EXIF UserComment prefixes
    if isinstance(comment, (bytes, bytearray)):
        try:
            s = comment.decode('utf-8', errors='ignore')
        except Exception:
            s = str(comment)
    else:
        s = str(comment)
    # Remove common ASCII prefix and NUL padding
    s = re.sub(r'(?i)^ASCII\x00*', '', s)
    s = s.strip('\x00').strip()

    # strict format: Yaw:...,Pitch:...,Roll:...
    m = re.search(r'(?i)\bYaw\s*[:=]\s*([-+]?\d+(?:\.\d+)?)\s*,\s*Pitch\s*[:=]\s*([-+]?\d+(?:\.\d+)?)\s*,\s*Roll\s*[:=]\s*([-+]?\d+(?:\.\d+)?)', s)
    if m:
        try:
            return (float(m.group(1)), float(m.group(2)), float(m.group(3)))
        except Exception:
            return (None, None, None)
    return (None, None, None)


def parse_user_comment_for_gps(comment: Any) -> tuple[float | None, float | None, float | None]:
    """
    Parse numeric GPS triple from UserComment if present.
    Returns (lat, lon, alt) floats or (None, None, None).
    """
    if comment is None:
        return (None, None, None)
    if isinstance(comment, (bytes, bytearray)):
        try:
            s = comment.decode('utf-8', errors='ignore')
        except Exception:
            s = str(comment)
    else:
        s = str(comment)
    s = s.strip('\x00').strip()

    m = re.search(r'([-+]?\d{1,3}\.\d+)[,\s]+([-+]?\d{1,3}\.\d+)[,\s]+([-+]?\d+(?:\.\d+)?)', s)
    if m:
        try:
            return (float(m.group(1)), float(m.group(2)), float(m.group(3)))
        except Exception:
            pass
    ml = re.search(r'lat\s*[:=]\s*([-+]?\d{1,3}\.\d+).*lon\s*[:=]\s*([-+]?\d{1,3}\.\d+).*alt\s*[:=]\s*([-+]?\d+(?:\.\d+)?)', s, flags=re.I)
    if ml:
        try:
            return (float(ml.group(1)), float(ml.group(2)), float(ml.group(3)))
        except Exception:
            pass
    return (None, None, None)


def compute_intrinsics(image_width: int, image_height: int, focal_length_mm: float | None, sensor_width_mm: float | None) -> dict[str, Any] | None:
    """
    Compute focal_pixels and intrinsic matrix K when possible.
    """
    if focal_length_mm is None or sensor_width_mm is None:
        return None
    fx = focal_length_mm * (image_width / sensor_width_mm)
    fy = fx
    cx = image_width / 2.0
    cy = image_height / 2.0
    K = [[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]]
    return {'focal_pixels': fx, 'K': K}


def extract_metadata_opencamera_style(image_path: str | Path) -> dict[str, Any]:
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'Image not found: {image_path}')

    img = Image.open(image_path)
    image_width, image_height = img.size

    exif_data = get_exif_dict(image_path)
    gps_exif = exif_data.get('GPSInfo', {}) if isinstance(exif_data.get('GPSInfo'), dict) else {}

    lat_ref = gps_exif.get('GPSLatitudeRef')
    lon_ref = gps_exif.get('GPSLongitudeRef')
    lat = dms_to_decimal(gps_exif.get('GPSLatitude'), lat_ref)
    lon = dms_to_decimal(gps_exif.get('GPSLongitude'), lon_ref)

    alt = rational_to_float(gps_exif.get('GPSAltitude'))
    # FIX: GPSAltitudeRef is stored as bytes (e.g. b'\x00' = above, b'\x01' = below sea level).
    # rational_to_float() returns None for bytes, so the sign was never applied.
    alt_ref_raw = gps_exif.get('GPSAltitudeRef')
    if isinstance(alt_ref_raw, (bytes, bytearray)):
        alt_ref_num = int.from_bytes(alt_ref_raw, 'big') if alt_ref_raw else 0
    else:
        alt_ref_num = rational_to_float(alt_ref_raw)
    if alt_ref_num == 1 and alt is not None:
        alt = -alt

    focal_length_mm = rational_to_float(exif_data.get('FocalLength'))
    focal_length_35mm_equiv = rational_to_float(exif_data.get('FocalLengthIn35mmFilm'))

    xmp_text = extract_xmp_text(image_path)
    xmp_attrs = parse_xmp_attributes(xmp_text)

    yaw = pick_float(
        [
            'PoseHeadingDegrees',
            'GPSImgDirection',
            'MAPCompassHeading',
            'CameraYaw',
            'Yaw',
            'yaw',
        ],
        xmp_attrs,
    )

    if yaw is None:
        yaw = rational_to_float(gps_exif.get('GPSImgDirection'))

    pitch = pick_float(
        [
            'MAPCapturePitch',
            'CameraPitch',
            'Pitch',
            'pitch',
            'PosePitchDegrees',
        ],
        xmp_attrs,
    )

    roll = pick_float(
        [
            'MAPCaptureRoll',
            'CameraRoll',
            'Roll',
            'roll',
            'PoseRollDegrees',
        ],
        xmp_attrs,
    )

    # Parse UserComment strictly (no guessing)
    if 'UserComment' in exif_data:
        yc, pc, rc = parse_user_comment_for_pose(exif_data.get('UserComment'))
        if yaw is None and yc is not None:
            yaw = yc
        if pitch is None and pc is not None:
            pitch = pc
        if roll is None and rc is not None:
            roll = rc

    # Some captures may put GPS in XMP instead of EXIF.
    if lat is None:
        lat = pick_float(['GPSLatitude', 'Latitude', 'MAPLatitude', 'lat'], xmp_attrs)
    if lon is None:
        lon = pick_float(['GPSLongitude', 'Longitude', 'MAPLongitude', 'lon'], xmp_attrs)
    if alt is None:
        alt = pick_float(['GPSAltitude', 'AbsoluteAltitude', 'RelativeAltitude', 'MAPAltitude'], xmp_attrs)

    if focal_length_mm is None:
        focal_length_mm = pick_float(['FocalLength', 'focal_length_mm'], xmp_attrs)

    if focal_length_35mm_equiv is None:
        focal_length_35mm_equiv = pick_float(
            ['FocalLengthIn35mmFilm', 'focal_length_35mm_equiv', 'FocalLength35mm'],
            xmp_attrs,
        )

    sensor_width_mm = pick_float(
        ['SensorWidth', 'sensor_width_mm', 'ExifSensorWidth', 'sensorWidth'],
        xmp_attrs,
    )
    sensor_height_mm = pick_float(
        ['SensorHeight', 'sensor_height_mm', 'ExifSensorHeight', 'sensorHeight'],
        xmp_attrs,
    )

    # Derive sensor size if missing and enough focal data exists.
    if sensor_width_mm is None and focal_length_mm is not None and focal_length_35mm_equiv is not None and focal_length_35mm_equiv > 0:
        sensor_width_mm = 36.0 * focal_length_mm / focal_length_35mm_equiv

    if sensor_height_mm is None and sensor_width_mm is not None:
        sensor_height_mm = sensor_width_mm * (float(image_height) / float(image_width))

    K_info = compute_intrinsics(image_width, image_height, focal_length_mm, sensor_width_mm)

    result = {
        'image_path': str(image_path),
        'yaw': yaw,
        'pitch': pitch,
        'roll': roll,
        'gps': {
            'latitude': lat,
            'longitude': lon,
            'altitude_m': alt,
        },
        'image_width': int(image_width),
        'image_height': int(image_height),
        'focal_length_mm': focal_length_mm,
        'sensor_width_mm': sensor_width_mm,
        'sensor_height_mm': sensor_height_mm,
        'focal_length_35mm_equiv': focal_length_35mm_equiv,
        'intrinsics': K_info,
        'metadata_sources': {
            'has_exif': bool(exif_data),
            'has_xmp': bool(xmp_text.strip()),
            'xmp_keys_count': len(xmp_attrs),
        },
    }

    return result


In [4]:
# Replace with your Mapillary-captured image path.
image_path = '/home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom/IMG_20260527_104328 (2).jpg'

meta = extract_metadata_opencamera_style(image_path)

print(json.dumps(meta, indent=2, ensure_ascii=False))

{
  "image_path": "/home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom/IMG_20260527_104328 (2).jpg",
  "yaw": 319.47,
  "pitch": 1.5786355125531224,
  "roll": -0.4576317215058907,
  "gps": {
    "latitude": 16.0757327,
    "longitude": 108.15537110000001,
    "altitude_m": 7.0
  },
  "image_width": 4000,
  "image_height": 3000,
  "focal_length_mm": 4.6,
  "sensor_width_mm": 6.624,
  "sensor_height_mm": 4.968,
  "focal_length_35mm_equiv": 25.0,
  "intrinsics": {
    "focal_pixels": 2777.777777777778,
    "K": [
      [
        2777.777777777778,
        0.0,
        2000.0
      ],
      [
        0.0,
        2777.777777777778,
        1500.0
      ],
      [
        0.0,
        0.0,
        1.0
      ]
    ]
  },
  "metadata_sources": {
    "has_exif": true,
    "has_xmp": false,
    "xmp_keys_count": 0
  }
}


## Notes

1. Some Mapillary exports keep orientation fields in XMP and GPS in EXIF, while others may keep both in one place.
2. `sensor_width_mm` and `sensor_height_mm` are often missing in consumer image metadata. This notebook estimates them when possible from focal length and 35mm equivalent focal length.
3. If `yaw/pitch/roll` come back as null, inspect `xmp_keys_count` and print XMP attributes for your specific device format.

In [5]:
# Debug raw EXIF IFDs from Pillow
from PIL import ExifTags, Image

img = Image.open(image_path)
exif = img.getexif()
print('Top-level tag count:', len(exif))
print('Top-level tag names:')
for tag_id, val in exif.items():
    name = ExifTags.TAGS.get(tag_id, str(tag_id))
    print('-', name, ':', type(val).__name__)

if hasattr(ExifTags, 'IFD'):
    for ifd_name in ['Exif', 'GPSInfo', 'Interop', 'MakerNote']:
        ifd_id = getattr(ExifTags.IFD, ifd_name, None)
        if ifd_id is None:
            continue
        try:
            ifd_data = exif.get_ifd(ifd_id)
            print(f'\n{ifd_name} IFD keys:', len(ifd_data) if ifd_data else 0)
            if ifd_data:
                for k, v in ifd_data.items():
                    name_map = ExifTags.GPSTAGS if ifd_name == 'GPSInfo' else ExifTags.TAGS
                    print('  -', name_map.get(k, str(k)), '=', v)
        except Exception as e:
            print(f'Cannot read {ifd_name} IFD:', e)

Top-level tag count: 13
Top-level tag names:
- ImageWidth : int
- ImageLength : int
- GPSInfo : int
- ResolutionUnit : int
- ExifOffset : int
- Make : str
- Model : str
- Software : str
- Orientation : int
- DateTime : str
- YCbCrPositioning : int
- XResolution : IFDRational
- YResolution : IFDRational

Exif IFD keys: 40
  - ExifVersion = 0220
  - SceneType = b'\x01\x00\x00\x00'
  - ApertureValue = 2.0
  - ColorSpace = 1
  - ExposureBiasValue = 0.0
  - MaxApertureValue = 2.0
  - ExifImageHeight = 3000
  - BrightnessValue = 2.03
  - DateTimeOriginal = 2026:05:27 10:43:28
  - FlashPixVersion = b'0100'
  - WhiteBalance = 0
  - ExifInteroperabilityOffset = 1218
  - Flash = 0
  - UserComment = ASCII   Yaw:319.4704,Pitch:1.5786355125531224,Roll:-0.4576317215058907
  - ExifImageWidth = 4000
  - Saturation = 0
  - OffsetTime = +07:00
  - SubsecTimeOriginal = 0073
  - SubsecTime = 0073
  - OffsetTimeDigitized = +07:00
  - ComponentsConfiguration = b'\x01\x02\x03\x00'
  - SubsecTimeDigitized = 0